# Projeto (Seção 11) — solução comentada

Continua a solução comentada da seção 10, organizada como a "solução final" do instrutor (imports → dados → tipos → base → modelos).
Enunciado em [`04_projeto`](04_projeto.ipynb).

Decisões que diferem da solução do instrutor:
- **`RegistrationCountry.name` é obrigatório.** "Sem país" já se expressa com `Automobile.registration_country = None`; um submodelo de
  país sem nome não tem significado e quebrava o `code3`.
- **`RegistrationCountry` é congelado (`frozen=True`).** Com a origem imutável, o `cached_property` de `code3` nunca fica desatualizado.
  Trocar o país passa a ser trocar o **objeto** inteiro (`car.registration_country = {"name": "france"}`), que é validado e tem cache novo.
  Isso substitui o `__setattr__` da seção 9 — composição + imutabilidade resolvem o problema de forma mais simples.
- Mantém o `CustomDate` simétrico, os `AliasChoices` e os validadores de data bidirecionais das seções anteriores.

#### Imports

In [1]:
import json
import re
from datetime import date, datetime
from enum import Enum
from functools import cached_property
from typing import Annotated, Any, TypeVar
from uuid import UUID, uuid4

from pydantic import (UUID4, AfterValidator, AliasChoices, BaseModel, BeforeValidator, ConfigDict, Field,
                      PlainSerializer, ValidationError, ValidationInfo, computed_field, field_validator)
from pydantic.alias_generators import to_camel

#### Dados

In [2]:
countries = {
    "australia": ("Australia", "AUS"), "canada": ("Canada", "CAN"), "china": ("China", "CHN"),
    "france": ("France", "FRA"), "germany": ("Germany", "DEU"), "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"), "norway": ("Norway", "NOR"), "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"), "sanmarino": ("San Marino", "SMR"), "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"), "united kingdom": ("United Kingdom", "GBR"), "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"), "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"), "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"), "united states of america": ("United States of America", "USA"),
    "brazil": ("Brazil", "BRA"), "brasil": ("Brazil", "BRA"),
}
VALID_COUNTRY_KEYS = sorted(countries)
country_code_lookup = {name: code for name, code in countries.values()}

#### Enums e funções auxiliares

In [3]:
class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


def lookup_country(name: str) -> tuple[str, str]:
    key = " ".join(name.split()).casefold()
    try:
        return countries[key]
    except KeyError:
        raise ValueError(f"unknown country {name!r}; expected one of: {', '.join(VALID_COUNTRY_KEYS)}") from None


DATE_FORMAT = "%Y/%m/%d"
SLASH_DATE = re.compile(r"\d{4}/\d{1,2}/\d{1,2}")


def parse_custom_date(value: Any) -> Any:
    if isinstance(value, str) and SLASH_DATE.fullmatch(value.strip()):
        try:
            return datetime.strptime(value.strip(), DATE_FORMAT).date()
        except ValueError as ex:
            raise ValueError(f"invalid date {value!r}: {ex}") from None
    return value


def serialize_custom_date(value: date) -> str:
    return value.strftime(DATE_FORMAT)


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")

#### Tipos `Annotated`

In [4]:
T = TypeVar("T")
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]
Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]
CustomDate = Annotated[
    date,
    BeforeValidator(parse_custom_date),
    PlainSerializer(serialize_custom_date, return_type=str, when_used="json-unless-none"),
]

#### Modelo base customizado

In [5]:
class CamelBaseModel(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

#### Modelos

In [6]:
class RegistrationCountry(CamelBaseModel):
    model_config = ConfigDict(frozen=True)          # mesclado com a config de CamelBaseModel

    name: Country

    @computed_field
    @cached_property
    def code3(self) -> str:
        return country_code_lookup[self.name]


class Automobile(CamelBaseModel):
    id_: UUID4 = Field(default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: CustomDate = Field(
        validation_alias=AliasChoices("completionDate", "manufacturedDate"), ge=date(1980, 1, 1), repr=False)
    base_msrp_usd: float = Field(
        validation_alias=AliasChoices("msrpUSD", "baseMSRPUSD"), serialization_alias="baseMSRPUSD", repr=False)
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, validation_alias=AliasChoices("doors", "numberOfDoors"), ge=2, le=4, multiple_of=2, repr=False)
    registration_country: RegistrationCountry | None = Field(default=None, repr=False)
    registration_date: CustomDate | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)

    @field_validator("manufactured_date")
    @classmethod
    def manufactured_not_after_registration(cls, value: date, info: ValidationInfo) -> date:
        registered = info.data.get("registration_date")
        if registered is not None and value > registered:
            raise ValueError(f"manufactured_date {value} is after registration_date {registered}")
        return value

    @field_validator("registration_date")
    @classmethod
    def registration_not_before_manufacture(cls, value: date | None, info: ValidationInfo) -> date | None:
        manufactured = info.data.get("manufactured_date")
        if value is not None and manufactured is not None and value < manufactured:
            raise ValueError(f"registration_date {value} is before manufactured_date {manufactured}")
        return value

## Teste do enunciado

In [7]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": {"name": "us"},
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': {
        'name': 'United States of America', 
        'code3': 'USA'
    },
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB'
}

car = Automobile.model_validate(data)
assert car.model_dump(by_alias=True) == expected_serialized_by_alias
print(f"car: {car!r}")
print(f"car.registration_country: {car.registration_country!r}")
print("teste do enunciado: ok")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>)
car.registration_country: RegistrationCountry(name='United States of America', code3='USA')
teste do enunciado: ok


## Testes extras

In [8]:
# a configuração foi herdada (e mesclada no submodelo)
print(f"RegistrationCountry.model_config['frozen']: {RegistrationCountry.model_config['frozen']}")
print(f"RegistrationCountry.model_config['extra']: {RegistrationCountry.model_config['extra']}")

# submodelo sem nome é rejeitado na validação (em vez de KeyError na serialização)
try:
    Automobile.model_validate(data | {"registrationCountry": {}})
except ValidationError as ex:
    print(f"país sem nome: {ex.errors()[0]['loc']} -> {ex.errors()[0]['msg']}")

# país ausente: None no dict e null no JSON
no_country = {k: v for k, v in data.items() if k != "registrationCountry"}
print(f"sem país (json): {json.loads(Automobile.model_validate(no_country).model_dump_json(by_alias=True))['registrationCountry']}")

# submodelo congelado: não dá para mutar o nome (logo o cache de code3 nunca fica velho)
car = Automobile.model_validate(data)
try:
    car.registration_country.name = "france"
except ValidationError as ex:
    print(f"mutar name: {ex.errors()[0]['type']}")

# trocar o país = trocar o objeto (validado por validate_assignment, cache novo)
car.registration_country = {"name": " Brasil "}
print(f"novo país: {car.registration_country!r}")
try:
    car.registration_country = {"name": "atlantis"}
except ValidationError as ex:
    print(f"país inválido na atribuição: {ex.errors()[0]['loc']}")

# extra no submodelo também é proibido (config herdada)
try:
    Automobile.model_validate(data | {"registrationCountry": {"name": "us", "capital": "DC"}})
except ValidationError as ex:
    print(f"extra no submodelo: {ex.errors()[0]['type']} em {ex.errors()[0]['loc']}")

# ida e volta pelo JSON (code3 é calculado: fica de fora porque extra="forbid")
car = Automobile.model_validate(data)
payload = car.model_dump_json(by_alias=True, exclude={"registration_country": {"code3"}})
assert Automobile.model_validate_json(payload) == car
print(f"payload['registrationCountry']: {json.loads(payload)['registrationCountry']}")
print("testes extras: ok")

RegistrationCountry.model_config['frozen']: True
RegistrationCountry.model_config['extra']: forbid
país sem nome: ('registrationCountry', 'name') -> Field required
sem país (json): None
mutar name: frozen_instance
novo país: RegistrationCountry(name='Brazil', code3='BRA')
país inválido na atribuição: ('registration_country', 'name')
extra no submodelo: extra_forbidden em ('registrationCountry', 'capital')
payload['registrationCountry']: {'name': 'United States of America'}
testes extras: ok


## Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| `CamelBaseModel` + herança da configuração | ✅ | ✅ (com `make_alias`, sem aliases manuais de `id`/`type`) |
| submodelo `RegistrationCountry` por composição | ✅ | ✅ |
| `name` | `Country \| None` → `{}` vira país sem nome e `KeyError` no dump | `Country` obrigatório → erro de validação claro |
| `code3` cacheado × `name` mutável | **cache desatualizado** | submodelo `frozen=True` |
| `TypeError` sem `registrationDate` | presente | corrigido (seção 8) |
| ida e volta pelo JSON | ❌ | ✅ (`exclude` aninhado de `code3`) |

> 💡 O `exclude` aninhado (`{"registration_country": {"code3"}}`) é a forma de tirar um campo de **dentro** de um submodelo na
> serialização. Para listas de submodelos, a chave é o índice ou `"__all__"`.